# CHINA'S REAL ESTATE DEMAND PREDICTION
## Kaggle Competition  

## 1.0 Datasets and Their Structure
1. train CSVs
    * new_house_transactions**
    * pre_owned_house_transactions
    * land_transactions
    * sector_POI
    * city_indexes
    * pre_owned_house_transactions_nearby_sectors
    * city_search_index
    * land_transactions_nearby_sectors
    * new_house_transactions_nearby_sectors
2. test CSV

In [1]:
#Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob, os
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import GridSearchCV
import joblib

In [171]:
#Load datasets
# NB: load new house transactions (target is here)
new_house = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/new_house_transactions.csv")

# Target variable
TARGET = "amount_new_house_transactions"

# Load all other relevant datasets
pre_owned = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/pre_owned_house_transactions.csv")
land = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/land_transactions.csv")
poi = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/sector_POI.csv")
city_indexes = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/city_indexes.csv")
pre_owned_nearby_sectors = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/pre_owned_house_transactions_nearby_sectors.csv")
city_search_index = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/city_search_index.csv")
land_nearby_sectors = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/land_transactions_nearby_sectors.csv")
new_house_nearby_sectors = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/train/new_house_transactions_nearby_sectors.csv")

#Load test data
test_df = pd.read_csv("C:/Users/USER/Downloads/china-real-estate-demand-prediction/test.csv")

In [172]:
# Peek at the new house transactions data
new_house.head()

,month,sector,num_new_house_transactions,area_new_house_transactions,price_new_house_transactions,amount_new_house_transactions,area_per_unit_new_house_transactions,total_price_per_unit_new_house_transactions,num_new_house_available_for_sale,area_new_house_available_for_sale,period_new_house_sell_through
0,2019-Jan,sector 1,52,4906,28184,13827.14,94,265.91,159.0,15904.0,3.78
1,2019-Jan,sector 2,145,15933,17747,28277.73,110,195.02,1491.0,175113.0,12.29
2,2019-Jan,sector 4,6,725,28004,1424.21,127,356.05,40.0,6826.0,5.95
3,2019-Jan,sector 5,2,212,37432,792.10,106,396.05,161.0,17173.0,83.95
4,2019-Jan,sector 6,5,773,15992,607.94,95,151.99,189.0,19696.0,14.27


In [173]:
new_house.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5433 entries, 0 to 5432
Data columns (total 11 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   month                                        5433 non-null   object 
 1   sector                                       5433 non-null   object 
 2   num_new_house_transactions                   5433 non-null   int64  
 3   area_new_house_transactions                  5433 non-null   int64  
 4   price_new_house_transactions                 5433 non-null   int64  
 5   amount_new_house_transactions                5433 non-null   float64
 6   area_per_unit_new_house_transactions         5433 non-null   int64  
 7   total_price_per_unit_new_house_transactions  5433 non-null   float64
 8   num_new_house_available_for_sale             5419 non-null   float64
 9   area_new_house_available_for_sale            5419 non-null   float64
 10  

In [174]:
new_house

,month,sector,num_new_house_transactions,area_new_house_transactions,price_new_house_transactions,amount_new_house_transactions,area_per_unit_new_house_transactions,total_price_per_unit_new_house_transactions,num_new_house_available_for_sale,area_new_house_available_for_sale,period_new_house_sell_through
0,2019-Jan,sector 1,52,4906,28184,13827.14,94,265.91,159.0,15904.0,3.78
1,2019-Jan,sector 2,145,15933,17747,28277.73,110,195.02,1491.0,175113.0,12.29
2,2019-Jan,sector 4,6,725,28004,1424.21,127,356.05,40.0,6826.0,5.95
3,2019-Jan,sector 5,2,212,37432,792.10,106,396.05,161.0,17173.0,83.95
4,2019-Jan,sector 6,5,773,15992,607.94,95,151.99,189.0,19696.0,14.27
...,...,...,...,...,...,...,...,...,...,...,...
5428,2024-Jul,sector 91,70,7921,40967,32450.06,113,463.57,2133.0,341192.0,51.82
5429,2024-Jul,sector 92,211,22084,13949,30804.74,105,145.99,5908.0,636696.0,34.76
5430,2024-Jul,sector 93,62,8136,27452,22335.30,131,360.25,1323.0,150862.0,27.74
5431,2024-Jul,sector 94,44,5078,26367,13389.41,115,304.30,2027.0,215821.0,38.62


In [175]:
new_house.columns

Index(['month', 'sector', 'num_new_house_transactions',
       'area_new_house_transactions', 'price_new_house_transactions',
       'amount_new_house_transactions', 'area_per_unit_new_house_transactions',
       'total_price_per_unit_new_house_transactions',
       'num_new_house_available_for_sale', 'area_new_house_available_for_sale',
       'period_new_house_sell_through'],
      dtype='object')

In [176]:
#The month column is string and YY-MM format, we need to split it into year and month
new_house[["year", "month"]] = new_house["month"].str.split("-", expand=True)
new_house["year"] = new_house["year"].astype(int)

#Making the Year the first column
new_house = new_house[["year", "month", 'sector', 'num_new_house_transactions',
       'area_new_house_transactions', 'price_new_house_transactions',
       'amount_new_house_transactions', 'area_per_unit_new_house_transactions',
       'total_price_per_unit_new_house_transactions',
       'num_new_house_available_for_sale', 'area_new_house_available_for_sale',
       'period_new_house_sell_through']]

In [177]:
#Viewing the test data
test_df.head()

,id,new_house_transaction_amount
0,2024 Aug_sector 1,NaN
1,2024 Aug_sector 2,NaN
2,2024 Aug_sector 3,NaN
3,2024 Aug_sector 4,NaN
4,2024 Aug_sector 5,NaN


In [178]:
#SPlitting the id column to get year, month and sector
test_df[["year", "rest"]] = test_df["id"].str.split(" ", n=1, expand=True)
test_df[["month", "sector"]] = test_df["rest"].str.split("_", n=1, expand=True)

# Clean types
test_df["year"] = test_df["year"].astype(int)

# Convert Month string ("Jan") -> numeric datetime
#test["month"] = pd.to_datetime(test["month"] + "-" + test["Year"].astype(str), format="%b-%Y")



# Drop helper
test_df = test_df.drop(columns=["rest"])
test_df.head()

,id,new_house_transaction_amount,year,month,sector
0,2024 Aug_sector 1,NaN,2024,Aug,sector 1
1,2024 Aug_sector 2,NaN,2024,Aug,sector 2
2,2024 Aug_sector 3,NaN,2024,Aug,sector 3
3,2024 Aug_sector 4,NaN,2024,Aug,sector 4
4,2024 Aug_sector 5,NaN,2024,Aug,sector 5


In [179]:
#Resort the columns
test_df = test_df[["id", "year", "month", "sector"]]
test_df.head()

,id,year,month,sector
0,2024 Aug_sector 1,2024,Aug,sector 1
1,2024 Aug_sector 2,2024,Aug,sector 2
2,2024 Aug_sector 3,2024,Aug,sector 3
3,2024 Aug_sector 4,2024,Aug,sector 4
4,2024 Aug_sector 5,2024,Aug,sector 5


In [180]:
#Merge the test data with new_house data to get the features
test_df = test_df.merge(new_house, on=["year", "month", "sector"], how="left")

In [181]:
test_df

,id,year,month,sector,num_new_house_transactions,area_new_house_transactions,price_new_house_transactions,amount_new_house_transactions,area_per_unit_new_house_transactions,total_price_per_unit_new_house_transactions,num_new_house_available_for_sale,area_new_house_available_for_sale,period_new_house_sell_through
0,2024 Aug_sector 1,2024,Aug,sector 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024 Aug_sector 2,2024,Aug,sector 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024 Aug_sector 3,2024,Aug,sector 3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024 Aug_sector 4,2024,Aug,sector 4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024 Aug_sector 5,2024,Aug,sector 5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1147,2025 Jul_sector 92,2025,Jul,sector 92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1148,2025 Jul_sector 93,2025,Jul,sector 93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1149,2025 Jul_sector 94,2025,Jul,sector 94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1150,2025 Jul_sector 95,2025,Jul,sector 95,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [182]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1152 entries, 0 to 1151
Data columns (total 13 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   id                                           1152 non-null   object 
 1   year                                         1152 non-null   int64  
 2   month                                        1152 non-null   object 
 3   sector                                       1152 non-null   object 
 4   num_new_house_transactions                   0 non-null      float64
 5   area_new_house_transactions                  0 non-null      float64
 6   price_new_house_transactions                 0 non-null      float64
 7   amount_new_house_transactions                0 non-null      float64
 8   area_per_unit_new_house_transactions         0 non-null      float64
 9   total_price_per_unit_new_house_transactions  0 non-null      float64
 10  

In [183]:
#Viewing the columns in all the datasets
print("Columns in pre_owned dataset: \n", pre_owned.columns)
print("\nColumns in land dataset: \n", land.columns)
print("\nColumns in poi dataset: \n", poi.columns)
print("\nColumns in city_indexes dataset: \n", city_indexes.columns)
print("\nColumns in pre_owned_nearby_sectors dataset: \n", pre_owned_nearby_sectors.columns)
print("\nColumns in city_search_index dataset: \n", city_search_index.columns)
print("\nColumns in land_nearby_sectors dataset: \n", land_nearby_sectors.columns)
print("\nColumns in new_house_nearby_sectors dataset: \n", new_house_nearby_sectors.columns)

Columns in pre_owned dataset: 
 Index(['month', 'sector', 'area_pre_owned_house_transactions',
       'amount_pre_owned_house_transactions',
       'num_pre_owned_house_transactions',
       'price_pre_owned_house_transactions'],
      dtype='object')

Columns in land dataset: 
 Index(['month', 'sector', 'num_land_transactions', 'construction_area',
       'planned_building_area', 'transaction_amount'],
      dtype='object')

Columns in poi dataset: 
 Index(['sector', 'sector_coverage', 'population_scale', 'residential_area',
       'office_building', 'commercial_area', 'resident_population',
       'office_population', 'number_of_shops', 'catering',
       ...
       'medical_health_rehabilitation_institution_dense',
       'medical_health_first_aid_center_dense',
       'medical_health_blood_donation_station_dense',
       'medical_health_disease_prevention_institution_dense',
       'medical_health_general_hospital_dense', 'medical_health_clinic_dense',
       'education_training_sc

In [184]:
pre_owned.head()

,month,sector,area_pre_owned_house_transactions,amount_pre_owned_house_transactions,num_pre_owned_house_transactions,price_pre_owned_house_transactions
0,2019-Jan,sector 35,548,33.200,6,605.839416
1,2019-Jan,sector 23,3376,9137.764,48,27066.836490
2,2019-Jan,sector 80,3804,8980.000,44,23606.729760
3,2019-Jan,sector 53,0,0.000,0,11301.439410
4,2019-Jan,sector 84,9941,51515.904,103,51821.651750


In [185]:
pre_owned.columns

Index(['month', 'sector', 'area_pre_owned_house_transactions',
       'amount_pre_owned_house_transactions',
       'num_pre_owned_house_transactions',
       'price_pre_owned_house_transactions'],
      dtype='object')

In [186]:
#Splitting the month column into year and month
pre_owned[["year", "month"]] = pre_owned["month"].str.split("-", expand=True)
pre_owned["year"] = pre_owned["year"].astype(int)

#Making the Year the first column
pre_owned = pre_owned[["year", "month", 'sector', 'area_pre_owned_house_transactions',
       'amount_pre_owned_house_transactions',
       'num_pre_owned_house_transactions',
       'price_pre_owned_house_transactions']]

In [187]:
pre_owned.head()

,year,month,sector,area_pre_owned_house_transactions,amount_pre_owned_house_transactions,num_pre_owned_house_transactions,price_pre_owned_house_transactions
0,2019,Jan,sector 35,548,33.200,6,605.839416
1,2019,Jan,sector 23,3376,9137.764,48,27066.836490
2,2019,Jan,sector 80,3804,8980.000,44,23606.729760
3,2019,Jan,sector 53,0,0.000,0,11301.439410
4,2019,Jan,sector 84,9941,51515.904,103,51821.651750


In [188]:
land.head()

,month,sector,num_land_transactions,construction_area,planned_building_area,transaction_amount
0,2019-Jan,sector 74,0,0.0,0.0,0.0
1,2019-Jan,sector 35,0,0.0,0.0,0.0
2,2019-Jan,sector 23,0,0.0,0.0,0.0
3,2019-Jan,sector 80,0,0.0,0.0,0.0
4,2019-Jan,sector 53,0,0.0,0.0,0.0


In [189]:
land.columns

Index(['month', 'sector', 'num_land_transactions', 'construction_area',
       'planned_building_area', 'transaction_amount'],
      dtype='object')

In [190]:
#Splitting the month column into year and month
land[["year", "month"]] = land["month"].str.split("-", expand=True)
land["year"] = land["year"].astype(int)
#Making the Year the first column
land = land[['year', 'month', 'sector', 'num_land_transactions', 'construction_area',
       'planned_building_area', 'transaction_amount']]
land.head()

,year,month,sector,num_land_transactions,construction_area,planned_building_area,transaction_amount
0,2019,Jan,sector 74,0,0.0,0.0,0.0
1,2019,Jan,sector 35,0,0.0,0.0,0.0
2,2019,Jan,sector 23,0,0.0,0.0,0.0
3,2019,Jan,sector 80,0,0.0,0.0,0.0
4,2019,Jan,sector 53,0,0.0,0.0,0.0


In [191]:
pre_owned_nearby_sectors.head()

,month,sector,num_pre_owned_house_transactions_nearby_sectors,area_pre_owned_house_transactions_nearby_sectors,amount_pre_owned_house_transactions_nearby_sectors,price_pre_owned_house_transactions_nearby_sectors
0,2019-Jan,sector 1,6.750000,733.000000,1247.03800,17012.79673
1,2019-Jan,sector 2,64.181818,5339.000000,20880.24282,39108.90208
2,2019-Jan,sector 3,77.714286,7457.142857,17376.21486,23301.43755
3,2019-Jan,sector 4,57.666667,5109.666667,19021.15267,37225.81904
4,2019-Jan,sector 5,45.428571,3763.500000,15800.74143,41984.16747


In [192]:
#SPlitting the month column in pre_owned_nearby_sectors to year and month
pre_owned_nearby_sectors[["year", "month"]] = pre_owned_nearby_sectors["month"].str.split("-", expand=True)
pre_owned_nearby_sectors["year"] = pre_owned_nearby_sectors["year"].astype(int)
pre_owned_nearby_sectors.head()


,month,sector,num_pre_owned_house_transactions_nearby_sectors,area_pre_owned_house_transactions_nearby_sectors,amount_pre_owned_house_transactions_nearby_sectors,price_pre_owned_house_transactions_nearby_sectors,year
0,Jan,sector 1,6.750000,733.000000,1247.03800,17012.79673,2019
1,Jan,sector 2,64.181818,5339.000000,20880.24282,39108.90208,2019
2,Jan,sector 3,77.714286,7457.142857,17376.21486,23301.43755,2019
3,Jan,sector 4,57.666667,5109.666667,19021.15267,37225.81904,2019
4,Jan,sector 5,45.428571,3763.500000,15800.74143,41984.16747,2019


In [193]:
#Making the Year the first column
pre_owned_nearby_sectors = pre_owned_nearby_sectors[['year', 'month', 'sector', 'num_pre_owned_house_transactions_nearby_sectors',
       'area_pre_owned_house_transactions_nearby_sectors',
       'amount_pre_owned_house_transactions_nearby_sectors',
       'price_pre_owned_house_transactions_nearby_sectors']]
pre_owned_nearby_sectors.head()

,year,month,sector,num_pre_owned_house_transactions_nearby_sectors,area_pre_owned_house_transactions_nearby_sectors,amount_pre_owned_house_transactions_nearby_sectors,price_pre_owned_house_transactions_nearby_sectors
0,2019,Jan,sector 1,6.750000,733.000000,1247.03800,17012.79673
1,2019,Jan,sector 2,64.181818,5339.000000,20880.24282,39108.90208
2,2019,Jan,sector 3,77.714286,7457.142857,17376.21486,23301.43755
3,2019,Jan,sector 4,57.666667,5109.666667,19021.15267,37225.81904
4,2019,Jan,sector 5,45.428571,3763.500000,15800.74143,41984.16747


In [194]:
city_search_index.head()

,month,keyword,source,search_volume
0,2019-Jan,买房,PC端,1914
1,2019-Jan,买房,移动端,2646
2,2019-Jan,二手房市场,PC端,192
3,2019-Jan,二手房市场,移动端,204
4,2019-Jan,公积金,PC端,9160


In [195]:
#SPlitting the month column of city_search_index to year and month
city_search_index[["year", "month"]] = city_search_index["month"].str.split("-", expand=True)
city_search_index["year"] = city_search_index["year"].astype(int)
#Making the Year the first column
city_search_index = city_search_index[['year', 'month', 'keyword', 'source', 'search_volume']]
city_search_index.head()

,year,month,keyword,source,search_volume
0,2019,Jan,买房,PC端,1914
1,2019,Jan,买房,移动端,2646
2,2019,Jan,二手房市场,PC端,192
3,2019,Jan,二手房市场,移动端,204
4,2019,Jan,公积金,PC端,9160


In [196]:
land_nearby_sectors.head()

,month,sector,num_land_transactions_nearby_sectors,construction_area_nearby_sectors,planned_building_area_nearby_sectors,transaction_amount_nearby_sectors
0,2019-Jan,sector 35,0.0,0.0,0.0,0.0
1,2019-Jan,sector 23,0.0,0.0,0.0,0.0
2,2019-Jan,sector 80,0.0,0.0,0.0,0.0
3,2019-Jan,sector 53,0.0,0.0,0.0,0.0
4,2019-Jan,sector 84,0.0,0.0,0.0,0.0


In [197]:
land_nearby_sectors.columns

Index(['month', 'sector', 'num_land_transactions_nearby_sectors',
       'construction_area_nearby_sectors',
       'planned_building_area_nearby_sectors',
       'transaction_amount_nearby_sectors'],
      dtype='object')

In [198]:
#Splitting the month column of land_nearby_sectors to year and month
land_nearby_sectors[["year", "month"]] = land_nearby_sectors["month"].str.split("-", expand=True)
land_nearby_sectors["year"] = land_nearby_sectors["year"].astype(int)
#Making the Year the first column
land_nearby_sectors = land_nearby_sectors[['year', 'month', 'sector', 'num_land_transactions_nearby_sectors',
       'construction_area_nearby_sectors',
       'planned_building_area_nearby_sectors',
       'transaction_amount_nearby_sectors']]
land_nearby_sectors.head()

,year,month,sector,num_land_transactions_nearby_sectors,construction_area_nearby_sectors,planned_building_area_nearby_sectors,transaction_amount_nearby_sectors
0,2019,Jan,sector 35,0.0,0.0,0.0,0.0
1,2019,Jan,sector 23,0.0,0.0,0.0,0.0
2,2019,Jan,sector 80,0.0,0.0,0.0,0.0
3,2019,Jan,sector 53,0.0,0.0,0.0,0.0
4,2019,Jan,sector 84,0.0,0.0,0.0,0.0


In [199]:
new_house_nearby_sectors.head()

,month,sector,num_new_house_transactions_nearby_sectors,area_new_house_transactions_nearby_sectors,price_new_house_transactions_nearby_sectors,amount_new_house_transactions_nearby_sectors,area_per_unit_new_house_transactions_nearby_sectors,total_price_per_unit_new_house_transactions_nearby_sectors,num_new_house_available_for_sale_nearby_sectors,area_new_house_available_for_sale_nearby_sectors,period_new_house_sell_through_nearby_sectors
0,2019-Jan,sector 35,129.250000,13212.500000,21172.85714,27974.637500,102.224371,216.438201,2526.750000,302828.50000,21.910000
1,2019-Jan,sector 23,27.400000,2822.400000,47592.19459,13432.421000,103.007299,490.234343,390.600000,47866.60000,11.150000
2,2019-Jan,sector 80,81.285714,8670.000000,25508.56484,22115.925710,106.660808,272.076415,1124.285714,131539.71430,10.467143
3,2019-Jan,sector 53,28.500000,3428.833333,39242.77451,13455.693330,120.309941,472.129591,350.500000,43073.66667,15.613333
4,2019-Jan,sector 84,8.857143,1304.428571,62359.84558,8134.396429,147.274193,918.399597,207.400000,35174.00000,26.246000


In [200]:
new_house_nearby_sectors.columns

Index(['month', 'sector', 'num_new_house_transactions_nearby_sectors',
       'area_new_house_transactions_nearby_sectors',
       'price_new_house_transactions_nearby_sectors',
       'amount_new_house_transactions_nearby_sectors',
       'area_per_unit_new_house_transactions_nearby_sectors',
       'total_price_per_unit_new_house_transactions_nearby_sectors',
       'num_new_house_available_for_sale_nearby_sectors',
       'area_new_house_available_for_sale_nearby_sectors',
       'period_new_house_sell_through_nearby_sectors'],
      dtype='object')

In [201]:
#Splitting the month column of nwe_house_nearby_sectors to year and month
new_house_nearby_sectors[["year", "month"]] = new_house_nearby_sectors["month"].str.split("-", expand=True)
new_house_nearby_sectors["year"] = new_house_nearby_sectors["year"].astype(int)
#Making the Year the first column
new_house_nearby_sectors = new_house_nearby_sectors[['year', 'month', 'sector', 'num_new_house_transactions_nearby_sectors',
       'area_new_house_transactions_nearby_sectors',
       'price_new_house_transactions_nearby_sectors',
       'amount_new_house_transactions_nearby_sectors',
       'area_per_unit_new_house_transactions_nearby_sectors',
       'total_price_per_unit_new_house_transactions_nearby_sectors',
       'num_new_house_available_for_sale_nearby_sectors',
       'area_new_house_available_for_sale_nearby_sectors',
       'period_new_house_sell_through_nearby_sectors']]
new_house_nearby_sectors.head()

,year,month,sector,num_new_house_transactions_nearby_sectors,area_new_house_transactions_nearby_sectors,price_new_house_transactions_nearby_sectors,amount_new_house_transactions_nearby_sectors,area_per_unit_new_house_transactions_nearby_sectors,total_price_per_unit_new_house_transactions_nearby_sectors,num_new_house_available_for_sale_nearby_sectors,area_new_house_available_for_sale_nearby_sectors,period_new_house_sell_through_nearby_sectors
0,2019,Jan,sector 35,129.250000,13212.500000,21172.85714,27974.637500,102.224371,216.438201,2526.750000,302828.50000,21.910000
1,2019,Jan,sector 23,27.400000,2822.400000,47592.19459,13432.421000,103.007299,490.234343,390.600000,47866.60000,11.150000
2,2019,Jan,sector 80,81.285714,8670.000000,25508.56484,22115.925710,106.660808,272.076415,1124.285714,131539.71430,10.467143
3,2019,Jan,sector 53,28.500000,3428.833333,39242.77451,13455.693330,120.309941,472.129591,350.500000,43073.66667,15.613333
4,2019,Jan,sector 84,8.857143,1304.428571,62359.84558,8134.396429,147.274193,918.399597,207.400000,35174.00000,26.246000


In [202]:
# Merge datasets to create a training set
train_df = (
    new_house
    .merge(pre_owned, on=["year","month","sector"], how="left")
    .merge(pre_owned_nearby_sectors, on=["year","month","sector"], how="left")
    .merge(land, on=["year","month","sector"], how="left")
    .merge(land_nearby_sectors, on=["year","month","sector"], how="left")
    .merge(new_house_nearby_sectors, on=["year","month","sector"], how="left")
    .merge(city_search_index, on=["year","month"], how="left")
    .merge(city_indexes,left_on="year", right_on="city_indicator_data_year", how="left")
    .merge(poi, on=["sector"], how="left")
)
train_df.head()

,year,month,sector,num_new_house_transactions,area_new_house_transactions,price_new_house_transactions,amount_new_house_transactions,area_per_unit_new_house_transactions,total_price_per_unit_new_house_transactions,num_new_house_available_for_sale,area_new_house_available_for_sale,period_new_house_sell_through,area_pre_owned_house_transactions,amount_pre_owned_house_transactions,num_pre_owned_house_transactions,price_pre_owned_house_transactions,num_pre_owned_house_transactions_nearby_sectors,area_pre_owned_house_transactions_nearby_sectors,amount_pre_owned_house_transactions_nearby_sectors,price_pre_owned_house_transactions_nearby_sectors,num_land_transactions,construction_area,planned_building_area,transaction_amount,num_land_transactions_nearby_sectors,construction_area_nearby_sectors,planned_building_area_nearby_sectors,transaction_amount_nearby_sectors,num_new_house_transactions_nearby_sectors,area_new_house_transactions_nearby_sectors,price_new_house_transactions_nearby_sectors,amount_new_house_transactions_nearby_sectors,area_per_unit_new_house_transactions_nearby_sectors,total_price_per_unit_new_house_transactions_nearby_sectors,num_new_house_available_for_sale_nearby_sectors,area_new_house_available_for_sale_nearby_sectors,period_new_house_sell_through_nearby_sectors,keyword,source,search_volume,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,2019,Jan,sector 1,52,4906,28184,13827.14,94,265.91,159.0,15904.0,3.78,9163.0,40994.7,111.0,44739.38666,6.75,733.0,1247.038,17012.79673,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.444444,3532.444444,51992.52013,18366.06889,119.969811,623.753283,350.25,49809.875,29.69625,买房,PC端,1914,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.0,0.0,0.0,0.0,0.000005,0.0,0.0,0.0,0.000243,0.000096,0.0,0.0,0.0,0.0,0.0,0.000027,0.000015,0.000014,0.0,0.000563,0.000032,0.0,0.0,0.0,0.000339,0.000113,0.0,0.0,8.600000e-07,0.000041,0.000038,0.000016,0.000028,0.000063,0.000014
1,2019,Jan,sector 1,52,4906,28184,13827.14,94,265.91,159.0,15904.0,3.78,9163.0,40994.7,111.0,44739.38666,6.75,733.0,1247.038,17012.79673,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.444444,3532.444444,51992.52013,18366.06889,119.969811,623.753283,350.25,49809.875,29.69625,买房,移动端,2646,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.0,0.0,0.0,0.0,0.000005,0.0,0.0,

In [203]:
#Merge test data with other datasets
test_df = (
    test_df
    .merge(pre_owned, on=["year","month","sector"], how="left")
    .merge(pre_owned_nearby_sectors, on=["year","month","sector"], how="left")
    .merge(land, on=["year","month","sector"], how="left")
    .merge(land_nearby_sectors, on=["year","month","sector"], how="left")
    .merge(new_house_nearby_sectors, on=["year","month","sector"], how="left")
    .merge(city_search_index, on=["year","month"], how="left")
    .merge(city_indexes,left_on="year", right_on="city_indicator_data_year", how="left")
    .merge(poi, on=["sector"], how="left")
)
test_df.head()

,id,year,month,sector,num_new_house_transactions,area_new_house_transactions,price_new_house_transactions,amount_new_house_transactions,area_per_unit_new_house_transactions,total_price_per_unit_new_house_transactions,num_new_house_available_for_sale,area_new_house_available_for_sale,period_new_house_sell_through,area_pre_owned_house_transactions,amount_pre_owned_house_transactions,num_pre_owned_house_transactions,price_pre_owned_house_transactions,num_pre_owned_house_transactions_nearby_sectors,area_pre_owned_house_transactions_nearby_sectors,amount_pre_owned_house_transactions_nearby_sectors,price_pre_owned_house_transactions_nearby_sectors,num_land_transactions,construction_area,planned_building_area,transaction_amount,num_land_transactions_nearby_sectors,construction_area_nearby_sectors,planned_building_area_nearby_sectors,transaction_amount_nearby_sectors,num_new_house_transactions_nearby_sectors,area_new_house_transactions_nearby_sectors,price_new_house_transactions_nearby_sectors,amount_new_house_transactions_nearby_sectors,area_per_unit_new_house_transactions_nearby_sectors,total_price_per_unit_new_house_transactions_nearby_sectors,num_new_house_available_for_sale_nearby_sectors,area_new_house_available_for_sale_nearby_sectors,period_new_house_sell_through_nearby_sectors,keyword,source,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,2024 Aug_sector 1,2024,Aug,sector 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.0,0.0,0.0,0.0,4.730000e-06,0.0,0.0,0.0,0.000243,0.000096,0.0,0.0,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,0.000014,0.0,0.000563,3.180000e-05,0.000000e+00,0.0,0.0,0.000339,1.126890e-04,0.0,0.0,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
1,2024 Aug_sector 2,2024,Aug,sector 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.0,0.0,0.0,0.0,7.670000e-08,0.0,0.0,0.0,0.000002,0.000000,0.0,0.0,0.000000,0.0,0.000000,7.670000e-08,1.530000e-07,0.000000,0.0,0.000004,7.670000e-08,7.670000e-08,0.0,

In [204]:
print(train_df.shape, test_df.shape)

(385140, 255) (1152, 256)


In [205]:
train_df.isnull().sum()

year                                                                  0
month                                                                 0
sector                                                                0
num_new_house_transactions                                            0
area_new_house_transactions                                           0
                                                                  ...  
medical_health_clinic_dense                                       35100
education_training_school_education_middle_school_dense           35100
education_training_school_education_primary_school_dense          35100
education_training_school_education_kindergarten_dense            35100
education_training_school_education_research_institution_dense    35100
Length: 255, dtype: int64

In [206]:
test_df.isnull().sum()

id                                                                   0
year                                                                 0
month                                                                0
sector                                                               0
num_new_house_transactions                                        1152
                                                                  ... 
medical_health_clinic_dense                                        120
education_training_school_education_middle_school_dense            120
education_training_school_education_primary_school_dense           120
education_training_school_education_kindergarten_dense             120
education_training_school_education_research_institution_dense     120
Length: 256, dtype: int64

In [207]:
#Determine columns with more than 120 missing values
cols_to_drop = test_df.columns[test_df.isnull().sum() > 120].to_list().copy()
cols_to_drop


['num_new_house_transactions',
 'area_new_house_transactions',
 'price_new_house_transactions',
 'amount_new_house_transactions',
 'area_per_unit_new_house_transactions',
 'total_price_per_unit_new_house_transactions',
 'num_new_house_available_for_sale',
 'area_new_house_available_for_sale',
 'period_new_house_sell_through',
 'area_pre_owned_house_transactions',
 'amount_pre_owned_house_transactions',
 'num_pre_owned_house_transactions',
 'price_pre_owned_house_transactions',
 'num_pre_owned_house_transactions_nearby_sectors',
 'area_pre_owned_house_transactions_nearby_sectors',
 'amount_pre_owned_house_transactions_nearby_sectors',
 'price_pre_owned_house_transactions_nearby_sectors',
 'num_land_transactions',
 'construction_area',
 'planned_building_area',
 'transaction_amount',
 'num_land_transactions_nearby_sectors',
 'construction_area_nearby_sectors',
 'planned_building_area_nearby_sectors',
 'transaction_amount_nearby_sectors',
 'num_new_house_transactions_nearby_sectors',
 'ar

In [208]:
#Droppin columns in test_df where sum of missing data is greater than 120
test_df = test_df.drop(columns=cols_to_drop)
test_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1152 entries, 0 to 1151
Columns: 143 entries, id to education_training_school_education_research_institution_dense
dtypes: float64(139), int64(1), object(3)
memory usage: 1.3+ MB


In [210]:
test_df

,id,year,month,sector,sector_coverage,population_scale,residential_area,office_building,commercial_area,resident_population,office_population,number_of_shops,catering,retail,hotel,transportation_station,education,leisure_and_entertainment,bus_station_cnt,subway_station_cnt,rentable_shops,leisure_entertainment_entertainment_venue_game_arcade,leisure_entertainment_entertainment_venue_party_house,leisure_entertainment_cultural_venue_cultural_palace,office_building_industrial_building_industrial_building,education_training_school_education_middle_school,education_training_school_education_primary_school,education_training_school_education_kindergarten,education_training_school_education_research_institution,medical_health,medical_health_specialty_hospital,medical_health_tcm_hospital,medical_health_physical_examination_institution,medical_health_veterinary_station,medical_health_pharmaceutical_healthcare,medical_health_rehabilitation_institution,medical_health_first_aid_center,medical_health_blood_donation_station,medical_health_disease_prevention_institution,medical_health_general_hospital,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,2024 Aug_sector 1,2024,Aug,sector 1,0.311873,134900.0,93.0,27.0,2.0,113000.0,31000.0,3294.0,1256.0,1407.0,167.0,336.0,181.0,283.0,27.0,2.0,295.0,62.0,34.0,32.0,0.0,36.0,66.0,147.0,32.0,1310.0,74.0,0.0,0.0,0.0,789.0,262.0,0.0,0.0,2.0,95.0,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.000000,0.000000,0.000000,0.0,4.730000e-06,0.000000e+00,0.000000e+00,0.0,0.000243,9.550000e-05,0.0,0.00000,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,1.380000e-05,0.0,5.634440e-04,3.180000e-05,0.000000e+00,0.000000e+00,0.0,3.393570e-04,1.126890e-04,0.000000e+00,0.000000e+00,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
1,2024 Aug_sector 2,2024,Aug,sector 2,0.556383,60205.0,29.0,1.0,2.0,58779.0,3202.0,1502.0,333.0,888.0,32.0,118.0,113.0,136.0,32.0,0.0,22.0,1.0,2.0,0.0,0.0,4.0,6.0,9.0,4.0,57.0,1.0,1.0,0.0,0.0,37.0,5.0,0.0,0.0,1.0,5.0,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.000000,0.000000,0.000000,0.0,7.670000e-08,0.000000e+00,0.000000e+00,0.0,0.000002,0.000000e+00,0.0,0.00000,0.000000,0.0,0.000000,7.670000e-08,1.530000e-07,0.000000e+00,0.0,4.370000e-06,7.670000e-08,7.6700

In [211]:
test_df.select_dtypes(include="object").columns

Index(['id', 'month', 'sector'], dtype='object')

In [214]:
cols_to_drop

['num_new_house_transactions',
 'area_new_house_transactions',
 'price_new_house_transactions',
 'amount_new_house_transactions',
 'area_per_unit_new_house_transactions',
 'total_price_per_unit_new_house_transactions',
 'num_new_house_available_for_sale',
 'area_new_house_available_for_sale',
 'period_new_house_sell_through',
 'area_pre_owned_house_transactions',
 'amount_pre_owned_house_transactions',
 'num_pre_owned_house_transactions',
 'price_pre_owned_house_transactions',
 'num_pre_owned_house_transactions_nearby_sectors',
 'area_pre_owned_house_transactions_nearby_sectors',
 'amount_pre_owned_house_transactions_nearby_sectors',
 'price_pre_owned_house_transactions_nearby_sectors',
 'num_land_transactions',
 'construction_area',
 'planned_building_area',
 'transaction_amount',
 'num_land_transactions_nearby_sectors',
 'construction_area_nearby_sectors',
 'planned_building_area_nearby_sectors',
 'transaction_amount_nearby_sectors',
 'num_new_house_transactions_nearby_sectors',
 'ar

In [219]:
train_df

,year,month,sector,num_new_house_transactions,area_new_house_transactions,price_new_house_transactions,amount_new_house_transactions,area_per_unit_new_house_transactions,total_price_per_unit_new_house_transactions,num_new_house_available_for_sale,area_new_house_available_for_sale,period_new_house_sell_through,area_pre_owned_house_transactions,amount_pre_owned_house_transactions,num_pre_owned_house_transactions,price_pre_owned_house_transactions,num_pre_owned_house_transactions_nearby_sectors,area_pre_owned_house_transactions_nearby_sectors,amount_pre_owned_house_transactions_nearby_sectors,price_pre_owned_house_transactions_nearby_sectors,num_land_transactions,construction_area,planned_building_area,transaction_amount,num_land_transactions_nearby_sectors,construction_area_nearby_sectors,planned_building_area_nearby_sectors,transaction_amount_nearby_sectors,num_new_house_transactions_nearby_sectors,area_new_house_transactions_nearby_sectors,price_new_house_transactions_nearby_sectors,amount_new_house_transactions_nearby_sectors,area_per_unit_new_house_transactions_nearby_sectors,total_price_per_unit_new_house_transactions_nearby_sectors,num_new_house_available_for_sale_nearby_sectors,area_new_house_available_for_sale_nearby_sectors,period_new_house_sell_through_nearby_sectors,keyword,source,search_volume,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,2019,Jan,sector 1,52,4906,28184,13827.14,94,265.91,159.0,15904.0,3.78,9163.0,40994.7,111.0,44739.38666,6.75,733.0,1247.038,17012.79673,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.444444,3532.444444,51992.52013,18366.06889,119.969811,623.753283,350.250000,49809.87500,29.696250,买房,PC端,1914,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.000000,0.000000,0.000000,0.0,0.000005,0.0,0.0,0.0,0.000243,0.000096,0.0,0.0,0.0,0.0,0.0,0.000027,0.000015,0.000014,0.0,0.000563,0.000032,0.0,0.000000,0.0,0.000339,0.000113,0.000000,0.000000,8.600000e-07,0.000041,0.000038,0.000016,0.000028,0.000063,0.000014
1,2019,Jan,sector 1,52,4906,28184,13827.14,94,265.91,159.0,15904.0,3.78,9163.0,40994.7,111.0,44739.38666,6.75,733.0,1247.038,17012.79673,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.444444,3532.444444,51992.52013,18366.06889,119.969811,623.753283,350.250000,49809.87500,29.696250,买房,移动端,2646,...,8.600000e-07,4.300000e-07,4.300000e-07,8.6000

In [271]:
cols_to_drop_clean = [c for c in cols_to_drop if c != TARGET]

In [272]:
train = train_df.drop(columns=cols_to_drop_clean).drop_duplicates()
train.shape

(5433, 143)

In [273]:
train.head()

,year,month,sector,amount_new_house_transactions,sector_coverage,population_scale,residential_area,office_building,commercial_area,resident_population,office_population,number_of_shops,catering,retail,hotel,transportation_station,education,leisure_and_entertainment,bus_station_cnt,subway_station_cnt,rentable_shops,leisure_entertainment_entertainment_venue_game_arcade,leisure_entertainment_entertainment_venue_party_house,leisure_entertainment_cultural_venue_cultural_palace,office_building_industrial_building_industrial_building,education_training_school_education_middle_school,education_training_school_education_primary_school,education_training_school_education_kindergarten,education_training_school_education_research_institution,medical_health,medical_health_specialty_hospital,medical_health_tcm_hospital,medical_health_physical_examination_institution,medical_health_veterinary_station,medical_health_pharmaceutical_healthcare,medical_health_rehabilitation_institution,medical_health_first_aid_center,medical_health_blood_donation_station,medical_health_disease_prevention_institution,medical_health_general_hospital,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,2019,Jan,sector 1,13827.14,0.311873,134900.0,93.0,27.0,2.0,113000.0,31000.0,3294.0,1256.0,1407.0,167.0,336.0,181.0,283.0,27.0,2.0,295.0,62.0,34.0,32.0,0.0,36.0,66.0,147.0,32.0,1310.0,74.0,0.0,0.0,0.0,789.0,262.0,0.0,0.0,2.0,95.0,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.0,0.0,0.0,0.0,4.730000e-06,0.0,0.0,0.0,0.000243,9.550000e-05,0.0,0.0,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,1.380000e-05,0.0,0.000563,3.180000e-05,0.000000e+00,0.0,0.0,0.000339,1.126890e-04,0.0,0.0,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
60,2019,Jan,sector 2,28277.73,0.556383,60205.0,29.0,1.0,2.0,58779.0,3202.0,1502.0,333.0,888.0,32.0,118.0,113.0,136.0,32.0,0.0,22.0,1.0,2.0,0.0,0.0,4.0,6.0,9.0,4.0,57.0,1.0,1.0,0.0,0.0,37.0,5.0,0.0,0.0,1.0,5.0,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.0,0.0,0.0,0.0,7.670000e-08,0.0,0.0,0.0,0.000002,0.000000e+00,0.0,0.0,0.000000,0.0,0.000000,7.670000e-08,1.530000e-07,0.000000e+00,0.0,0.000004,7.670000e-08,7.670000e-08,0.0,0.0,0.000003,3.840000e-07,0.0,0.0,7.670000e-08,3.840000e-07,5.370000e-07,3.070000e-07,4.6000

In [274]:

#Handle missing values & select features
# ==========================================
# Drop identifiers and target for training features
target = "amount_new_house_transactions"
X = train.drop(columns=[target])
y = train[target]

# Basic preprocessing: fill NA
X = X.fillna(0)

In [275]:
X

,year,month,sector,sector_coverage,population_scale,residential_area,office_building,commercial_area,resident_population,office_population,number_of_shops,catering,retail,hotel,transportation_station,education,leisure_and_entertainment,bus_station_cnt,subway_station_cnt,rentable_shops,leisure_entertainment_entertainment_venue_game_arcade,leisure_entertainment_entertainment_venue_party_house,leisure_entertainment_cultural_venue_cultural_palace,office_building_industrial_building_industrial_building,education_training_school_education_middle_school,education_training_school_education_primary_school,education_training_school_education_kindergarten,education_training_school_education_research_institution,medical_health,medical_health_specialty_hospital,medical_health_tcm_hospital,medical_health_physical_examination_institution,medical_health_veterinary_station,medical_health_pharmaceutical_healthcare,medical_health_rehabilitation_institution,medical_health_first_aid_center,medical_health_blood_donation_station,medical_health_disease_prevention_institution,medical_health_general_hospital,medical_health_clinic,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,2019,Jan,sector 1,0.311873,134900.0,93.0,27.0,2.0,113000.0,31000.0,3294.0,1256.0,1407.0,167.0,336.0,181.0,283.0,27.0,2.0,295.0,62.0,34.0,32.0,0.0,36.0,66.0,147.0,32.0,1310.0,74.0,0.0,0.0,0.0,789.0,262.0,0.0,0.0,2.0,95.0,88.0,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.000000,0.000000,0.000000,0.0,4.730000e-06,0.000000e+00,0.000000e+00,0.0,0.000243,9.550000e-05,0.0,0.0,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,1.380000e-05,0.0,5.634440e-04,3.180000e-05,0.000000e+00,0.000000e+00,0.0,3.393570e-04,1.126890e-04,0.000000e+00,0.000000e+00,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
60,2019,Jan,sector 2,0.556383,60205.0,29.0,1.0,2.0,58779.0,3202.0,1502.0,333.0,888.0,32.0,118.0,113.0,136.0,32.0,0.0,22.0,1.0,2.0,0.0,0.0,4.0,6.0,9.0,4.0,57.0,1.0,1.0,0.0,0.0,37.0,5.0,0.0,0.0,1.0,5.0,7.0,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.000000,0.000000,0.000000,0.0,7.670000e-08,0.000000e+00,0.000000e+00,0.0,0.000002,0.000000e+00,0.0,0.0,0.000000,0.0,0.000000,7.670000e-08,1.530000e-07,0.000000e+00,0.0,4.370000e-06,7.670000e-08,7.670000e-08,0.000000

In [276]:
# Encode categorical (like sector) simply with factorize
#for col in X.select_dtypes(include="object").columns:
    #X[col], _ = pd.factorize(X[col])

#print("Feature matrix shape:", X.shape)
X = X.select_dtypes(exclude="object").drop(columns=["year"])
X.head()

,sector_coverage,population_scale,residential_area,office_building,commercial_area,resident_population,office_population,number_of_shops,catering,retail,hotel,transportation_station,education,leisure_and_entertainment,bus_station_cnt,subway_station_cnt,rentable_shops,leisure_entertainment_entertainment_venue_game_arcade,leisure_entertainment_entertainment_venue_party_house,leisure_entertainment_cultural_venue_cultural_palace,office_building_industrial_building_industrial_building,education_training_school_education_middle_school,education_training_school_education_primary_school,education_training_school_education_kindergarten,education_training_school_education_research_institution,medical_health,medical_health_specialty_hospital,medical_health_tcm_hospital,medical_health_physical_examination_institution,medical_health_veterinary_station,medical_health_pharmaceutical_healthcare,medical_health_rehabilitation_institution,medical_health_first_aid_center,medical_health_blood_donation_station,medical_health_disease_prevention_institution,medical_health_general_hospital,medical_health_clinic,transportation_facilities_service_bus_station,transportation_facilities_service_subway_station,transportation_facilities_service_airport_related,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,0.311873,134900.0,93.0,27.0,2.0,113000.0,31000.0,3294.0,1256.0,1407.0,167.0,336.0,181.0,283.0,27.0,2.0,295.0,62.0,34.0,32.0,0.0,36.0,66.0,147.0,32.0,1310.0,74.0,0.0,0.0,0.0,789.0,262.0,0.0,0.0,2.0,95.0,88.0,564.0,222.0,0.0,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.0,0.0,0.0,0.0,4.730000e-06,0.0,0.0,0.0,0.000243,9.550000e-05,0.0,0.0,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,1.380000e-05,0.0,0.000563,3.180000e-05,0.000000e+00,0.0,0.0,0.000339,1.126890e-04,0.0,0.0,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
60,0.556383,60205.0,29.0,1.0,2.0,58779.0,3202.0,1502.0,333.0,888.0,32.0,118.0,113.0,136.0,32.0,0.0,22.0,1.0,2.0,0.0,0.0,4.0,6.0,9.0,4.0,57.0,1.0,1.0,0.0,0.0,37.0,5.0,0.0,0.0,1.0,5.0,7.0,23.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.0,0.0,0.0,0.0,7.670000e-08,0.0,0.0,0.0,0.000002,0.000000e+00,0.0,0.0,0.000000,0.0,0.000000,7.670000e-08,1.530000e-07,0.000000e+00,0.0,0.000004,7.670000e-08,7.6700

In [277]:
test = test_df.select_dtypes(exclude="object").drop(columns=["year"])
test

,sector_coverage,population_scale,residential_area,office_building,commercial_area,resident_population,office_population,number_of_shops,catering,retail,hotel,transportation_station,education,leisure_and_entertainment,bus_station_cnt,subway_station_cnt,rentable_shops,leisure_entertainment_entertainment_venue_game_arcade,leisure_entertainment_entertainment_venue_party_house,leisure_entertainment_cultural_venue_cultural_palace,office_building_industrial_building_industrial_building,education_training_school_education_middle_school,education_training_school_education_primary_school,education_training_school_education_kindergarten,education_training_school_education_research_institution,medical_health,medical_health_specialty_hospital,medical_health_tcm_hospital,medical_health_physical_examination_institution,medical_health_veterinary_station,medical_health_pharmaceutical_healthcare,medical_health_rehabilitation_institution,medical_health_first_aid_center,medical_health_blood_donation_station,medical_health_disease_prevention_institution,medical_health_general_hospital,medical_health_clinic,transportation_facilities_service_bus_station,transportation_facilities_service_subway_station,transportation_facilities_service_airport_related,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,0.311873,134900.0,93.0,27.0,2.0,113000.0,31000.0,3294.0,1256.0,1407.0,167.0,336.0,181.0,283.0,27.0,2.0,295.0,62.0,34.0,32.0,0.0,36.0,66.0,147.0,32.0,1310.0,74.0,0.0,0.0,0.0,789.0,262.0,0.0,0.0,2.0,95.0,88.0,564.0,222.0,0.0,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.000000,0.000000,0.000000,0.0,4.730000e-06,0.000000e+00,0.000000e+00,0.0,0.000243,9.550000e-05,0.0,0.00000,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,1.380000e-05,0.0,5.634440e-04,3.180000e-05,0.000000e+00,0.000000e+00,0.0,3.393570e-04,1.126890e-04,0.000000e+00,0.000000e+00,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
1,0.556383,60205.0,29.0,1.0,2.0,58779.0,3202.0,1502.0,333.0,888.0,32.0,118.0,113.0,136.0,32.0,0.0,22.0,1.0,2.0,0.0,0.0,4.0,6.0,9.0,4.0,57.0,1.0,1.0,0.0,0.0,37.0,5.0,0.0,0.0,1.0,5.0,7.0,23.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.000000,0.000000,0.000000,0.0,7.670000e-08,0.000000e+00,0.000000e+00,0.0,0.000002,0.000000e+

In [278]:
#Handle missing values in test set
test = test.fillna(0)
#Encode categorical (like sector) simply with factorize
#for col in test.select_dtypes(include="object").columns:
    #test[col], _ = pd.factorize(test[col])
#print("Test feature matrix shape:", test.shape)

In [279]:
test.head()

,sector_coverage,population_scale,residential_area,office_building,commercial_area,resident_population,office_population,number_of_shops,catering,retail,hotel,transportation_station,education,leisure_and_entertainment,bus_station_cnt,subway_station_cnt,rentable_shops,leisure_entertainment_entertainment_venue_game_arcade,leisure_entertainment_entertainment_venue_party_house,leisure_entertainment_cultural_venue_cultural_palace,office_building_industrial_building_industrial_building,education_training_school_education_middle_school,education_training_school_education_primary_school,education_training_school_education_kindergarten,education_training_school_education_research_institution,medical_health,medical_health_specialty_hospital,medical_health_tcm_hospital,medical_health_physical_examination_institution,medical_health_veterinary_station,medical_health_pharmaceutical_healthcare,medical_health_rehabilitation_institution,medical_health_first_aid_center,medical_health_blood_donation_station,medical_health_disease_prevention_institution,medical_health_general_hospital,medical_health_clinic,transportation_facilities_service_bus_station,transportation_facilities_service_subway_station,transportation_facilities_service_airport_related,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,0.311873,134900.0,93.0,27.0,2.0,113000.0,31000.0,3294.0,1256.0,1407.0,167.0,336.0,181.0,283.0,27.0,2.0,295.0,62.0,34.0,32.0,0.0,36.0,66.0,147.0,32.0,1310.0,74.0,0.0,0.0,0.0,789.0,262.0,0.0,0.0,2.0,95.0,88.0,564.0,222.0,0.0,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.0,0.0,0.0,0.0,4.730000e-06,0.0,0.0,0.0,0.000243,0.000096,0.0,0.0,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,0.000014,0.0,0.000563,3.180000e-05,0.000000e+00,0.0,0.0,0.000339,1.126890e-04,0.0,0.0,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
1,0.556383,60205.0,29.0,1.0,2.0,58779.0,3202.0,1502.0,333.0,888.0,32.0,118.0,113.0,136.0,32.0,0.0,22.0,1.0,2.0,0.0,0.0,4.0,6.0,9.0,4.0,57.0,1.0,1.0,0.0,0.0,37.0,5.0,0.0,0.0,1.0,5.0,7.0,23.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.0,0.0,0.0,0.0,7.670000e-08,0.0,0.0,0.0,0.000002,0.000000,0.0,0.0,0.000000,0.0,0.000000,7.670000e-08,1.530000e-07,0.000000,0.0,0.000004,7.670000e-08,7.670000e-08,0.0,0.0,0.

In [280]:
X.head()

,sector_coverage,population_scale,residential_area,office_building,commercial_area,resident_population,office_population,number_of_shops,catering,retail,hotel,transportation_station,education,leisure_and_entertainment,bus_station_cnt,subway_station_cnt,rentable_shops,leisure_entertainment_entertainment_venue_game_arcade,leisure_entertainment_entertainment_venue_party_house,leisure_entertainment_cultural_venue_cultural_palace,office_building_industrial_building_industrial_building,education_training_school_education_middle_school,education_training_school_education_primary_school,education_training_school_education_kindergarten,education_training_school_education_research_institution,medical_health,medical_health_specialty_hospital,medical_health_tcm_hospital,medical_health_physical_examination_institution,medical_health_veterinary_station,medical_health_pharmaceutical_healthcare,medical_health_rehabilitation_institution,medical_health_first_aid_center,medical_health_blood_donation_station,medical_health_disease_prevention_institution,medical_health_general_hospital,medical_health_clinic,transportation_facilities_service_bus_station,transportation_facilities_service_subway_station,transportation_facilities_service_airport_related,...,commercial_buildings_dense,hypermarkets_dense,department_stores_dense,shopping_centers_dense,hotel_commercial_dense,third_tier_shopping_malls_in_business_district_dense,second_tier_shopping_malls_in_business_district_dense,city_winner_malls_dense,shopping_malls_with_street_facing_shops_dense,unranked_malls_dense,community_malls_dense,community_winner_malls_dense,key_focus_malls_dense,transportation_facilities_service_bus_station_dense,transportation_facilities_service_subway_station_dense,transportation_facilities_service_airport_related_dense,transportation_facilities_service_port_terminal_dense,transportation_facilities_service_train_station_dense,transportation_facilities_service_light_rail_station_dense,transportation_facilities_service_long_distance_bus_station_dense,leisure_entertainment_entertainment_venue_game_arcade_dense,leisure_entertainment_entertainment_venue_party_house_dense,leisure_entertainment_cultural_venue_cultural_palace_dense,office_building_industrial_building_industrial_building_dense,medical_health_dense,medical_health_specialty_hospital_dense,medical_health_tcm_hospital_dense,medical_health_physical_examination_institution_dense,medical_health_veterinary_station_dense,medical_health_pharmaceutical_healthcare_dense,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,0.311873,134900.0,93.0,27.0,2.0,113000.0,31000.0,3294.0,1256.0,1407.0,167.0,336.0,181.0,283.0,27.0,2.0,295.0,62.0,34.0,32.0,0.0,36.0,66.0,147.0,32.0,1310.0,74.0,0.0,0.0,0.0,789.0,262.0,0.0,0.0,2.0,95.0,88.0,564.0,222.0,0.0,...,8.600000e-07,4.300000e-07,4.300000e-07,8.600000e-07,0.0,0.0,0.0,0.0,0.0,4.730000e-06,0.0,0.0,0.0,0.000243,9.550000e-05,0.0,0.0,0.000000,0.0,0.000000,2.670000e-05,1.460000e-05,1.380000e-05,0.0,0.000563,3.180000e-05,0.000000e+00,0.0,0.0,0.000339,1.126890e-04,0.0,0.0,8.600000e-07,4.090000e-05,3.780000e-05,1.550000e-05,2.840000e-05,6.320000e-05,1.380000e-05
60,0.556383,60205.0,29.0,1.0,2.0,58779.0,3202.0,1502.0,333.0,888.0,32.0,118.0,113.0,136.0,32.0,0.0,22.0,1.0,2.0,0.0,0.0,4.0,6.0,9.0,4.0,57.0,1.0,1.0,0.0,0.0,37.0,5.0,0.0,0.0,1.0,5.0,7.0,23.0,0.0,0.0,...,0.000000e+00,0.000000e+00,0.000000e+00,7.670000e-08,0.0,0.0,0.0,0.0,0.0,7.670000e-08,0.0,0.0,0.0,0.000002,0.000000e+00,0.0,0.0,0.000000,0.0,0.000000,7.670000e-08,1.530000e-07,0.000000e+00,0.0,0.000004,7.670000e-08,7.6700

In [281]:
y.isna().any()

np.False_

In [282]:
#SSet the X_train and y_train
X_train = X
y_train = y
X_test = test

In [283]:
X_train.shape

(5433, 139)

In [284]:
y_train.shape

(5433,)

In [285]:
X_test.shape

(1152, 139)

In [286]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)


In [287]:
# Train XGBoost
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist"
)

xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=50)

[0]	validation_0-rmse:45938.03387
[50]	validation_0-rmse:36985.38681
[100]	validation_0-rmse:37026.12242
[150]	validation_0-rmse:37050.46168
[200]	validation_0-rmse:37011.27423
[250]	validation_0-rmse:37069.20815
[300]	validation_0-rmse:37009.50190
[350]	validation_0-rmse:37098.23029
[400]	validation_0-rmse:36994.36866
[450]	validation_0-rmse:37043.41199
[499]	validation_0-rmse:37049.76275


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [288]:
#Predict model on validation
val_preds = xgb_model.predict(X_val)
rmse = (root_mean_squared_error(y_val, val_preds))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 37049.7629


In [289]:
#Predict model on test
test_preds = xgb_model.predict(X_test)

In [290]:
submission2 = test_df[["id"]].copy()
submission2["new_house_transaction_amount"] = test_preds
submission2.head()

,id,new_house_transaction_amount
0,2024 Aug_sector 1,25842.373047
1,2024 Aug_sector 2,12555.369141
2,2024 Aug_sector 3,25917.417969
3,2024 Aug_sector 4,75543.015625
4,2024 Aug_sector 5,1798.687866


In [291]:
#SUbmission file
submission2.to_csv("submission3.csv", index=False)

In [292]:
submission2.shape

(1152, 2)

In [86]:
new_house

,year,month,sector,amount_new_house_transactions
0,2019,Jan,sector 1,13827.14
1,2019,Jan,sector 2,28277.73
2,2019,Jan,sector 4,1424.21
3,2019,Jan,sector 5,792.10
4,2019,Jan,sector 6,607.94
...,...,...,...,...
5428,2024,Jul,sector 91,32450.06
5429,2024,Jul,sector 92,30804.74
5430,2024,Jul,sector 93,22335.30
5431,2024,Jul,sector 94,13389.41
